# Build a Smart AI-Powered Search and Recommendation System Using AWS AI Services
### Notebook-Based Implementation (SageMaker Notebook Edition)

**Original lab format:** AWS Management Console (click-through)
**This notebook:** Fully code-based equivalent using `boto3`, runnable end-to-end inside a SageMaker Notebook Instance.

---

## Real-Time Scenario

You are a Machine Learning Engineer at a large retail company. Your task is to build an **Intelligent Business Assistant** that can:

1. **Search** through company documents using natural language (Amazon Kendra)
2. **Recommend** products to customers based on behavior (Amazon Personalize)
3. **Route low-confidence AI decisions to a human reviewer** before they are trusted (Amazon Augmented AI / A2I)

All of this is glued together with **Amazon S3** for storage and **AWS IAM** for secure, scoped permissions — mirroring real enterprise AI architectures used by companies like Amazon, Walmart, and Flipkart.

## What This Notebook Does

This notebook recreates **every console step** from the original lab guide as executable Python code:

| Section | Console Action | Notebook Equivalent |
|---|---|---|
| 1 | Create S3 bucket + folders | `boto3` S3 client calls |
| 2 | Upload documents/datasets | `upload_file()` / `put_object()` |
| 3 | Configure bucket policy | `put_bucket_policy()` |
| 4 | Create IAM roles | `boto3` IAM client + trust policies |
| 5 | Create Kendra index + S3 data source | Kendra `boto3` client |
| 6 | Sync & search Kendra | `start_data_source_sync_job`, `query` |
| 7 | Create Personalize dataset group | Personalize `boto3` client |
| 8 | Import interactions, train recommender | `create_dataset_import_job`, `create_recommender` |
| 9 | Get recommendations | `get_recommendations` (Personalize Runtime) |
| 10 | Create A2I human review workflow | SageMaker `boto3` client (A2I APIs) |
| 11 | Clean up all resources | Deletion calls for every resource created |

> **Beginner Note:** Every code cell below includes inline comments. Run cells **top to bottom, in order** — later cells depend on resources (bucket name, ARNs, role ARNs) created in earlier cells via Python variables, so you do not need to copy/paste ARNs manually like in the console.

---

## ⚠️ Cost & Cleanup Reminder

This notebook creates **real, billable AWS resources**:
- Amazon Kendra Developer Edition index — **billed hourly**, even when idle
- Amazon Personalize recommender — billed for training + hosting
- Amazon A2I workflow — billed per object reviewed (low cost), but the SageMaker Notebook instance itself bills hourly

**Run the Cleanup section (last section) as soon as you're done**, exactly as the original guide stresses in its "Delete And Clean Up" chapter. A verification + deletion cell is provided for every resource.

Estimated cost if completed and cleaned up promptly: **$0.00 – $1.00** (covered by AWS Free Tier credits in most cases), matching the original guide's estimate.


---
## Section 0: Environment Setup

Before creating any AWS resource, we set up our SDK clients and helper variables. This replaces opening the AWS Console and navigating between services — instead we instantiate one `boto3` client per service.

**Run this cell first, every time you restart the notebook.**


In [1]:
# Core imports used throughout the notebook
import boto3
import json
import time
import uuid
from datetime import datetime

# ---------------------------------------------------------------
# AWS Region & Account Setup
# ---------------------------------------------------------------
# This mirrors selecting your region in the top-right of the AWS Console.
REGION = "us-east-1"  # change if your lab requires a different region

# boto3 Session - reuses the IAM role/credentials attached to this
# SageMaker Notebook instance (no access keys need to be hardcoded).
session = boto3.Session(region_name=REGION)

# STS client lets us fetch our AWS Account ID programmatically,
# which we need to build globally-unique S3 bucket names and ARNs.
sts_client = session.client("sts")
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

print(f"AWS Account ID : {ACCOUNT_ID}")
print(f"AWS Region     : {REGION}")


# ---------------------------------------------------------------
# Safety guard: if variables are missing (e.g. after kernel restart),
# this block reminds you to re-run from Section 0.
# ---------------------------------------------------------------
_required = ["REGION", "ACCOUNT_ID", "session"]
_missing  = [v for v in _required if v not in dir()]
if _missing:
    print(f"WARNING: Missing variables {_missing}. Re-run Section 0 cells first.")
else:
    print(f"AWS Account ID : {ACCOUNT_ID}")
    print(f"AWS Region     : {REGION}")


AWS Account ID : 745404749079
AWS Region     : us-east-1
AWS Account ID : 745404749079
AWS Region     : us-east-1


In [2]:
# ---------------------------------------------------------------
# Service Clients
# ---------------------------------------------------------------
# Each of these replaces "searching for <service> in the console search bar"
s3_client       = session.client("s3")
iam_client      = session.client("iam")
kendra_client   = session.client("kendra")
personalize_client          = session.client("personalize")
personalize_runtime_client  = session.client("personalize-runtime")
sagemaker_client = session.client("sagemaker")  # used for Amazon A2I (A2I APIs live under SageMaker)

print("All AWS service clients initialized successfully.")


All AWS service clients initialized successfully.


In [3]:
# ---------------------------------------------------------------
# Global naming variables (fill in as you go / used throughout)
# ---------------------------------------------------------------
# A unique suffix avoids bucket name collisions, since S3 bucket names
# must be globally unique across ALL AWS accounts.
UNIQUE_SUFFIX = str(uuid.uuid4())[:8]

BUCKET_NAME = f"aws-ai-services-{ACCOUNT_ID}-{REGION}-{UNIQUE_SUFFIX}"

# Folder ("prefix") names inside the bucket - equivalent to the
# "Create folder" console steps for kendra-docs, personalize-data, and a2i-output
KENDRA_PREFIX      = "kendra-docs/"
PERSONALIZE_PREFIX = "personalize-data/"
A2I_OUTPUT_PREFIX  = "a2i-output/"

print(f"Planned S3 bucket name: {BUCKET_NAME}")
print(f"Folders to create     : {KENDRA_PREFIX}, {PERSONALIZE_PREFIX}, {A2I_OUTPUT_PREFIX}")


Planned S3 bucket name: aws-ai-services-745404749079-us-east-1-49f4e565
Folders to create     : kendra-docs/, personalize-data/, a2i-output/


---
## Section 1: Create and Configure the Amazon S3 Bucket

**Console steps being replaced:**
- Create bucket → General purpose, default settings
- Uncheck "Block all public access" (only needed for specific demo cases — see note below)
- Create folders: `kendra-docs/`, `personalize-data/`, `a2i-output/`
- Upload `return-policy.txt` into `kendra-docs/`
- Upload `interactions.csv` into `personalize-data/`
- Edit bucket policy to allow `personalize.amazonaws.com` access

> **Security Note:** The original console guide unchecks "Block all public access" for simplicity. In this notebook, we **do NOT make the bucket public** — we instead grant access only to the specific AWS service principals (`personalize.amazonaws.com`, `kendra.amazonaws.com`) that need it, which is the AWS-recommended, least-privilege approach. This achieves the same functional outcome (Kendra and Personalize can read the bucket) without exposing data publicly.


In [4]:
# ---------------------------------------------------------------
# Create the S3 bucket
# ---------------------------------------------------------------
# Equivalent console action: "Create bucket" with General Purpose type
try:
    if REGION == "us-east-1":
        # us-east-1 is the default region and does NOT accept a
        # LocationConstraint parameter (a common boto3 gotcha)
        s3_client.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={"LocationConstraint": REGION}
        )
    print(f"Bucket '{BUCKET_NAME}' created successfully.")
except Exception as e:
    print(f"Error creating bucket: {e}")


Bucket 'aws-ai-services-745404749079-us-east-1-49f4e565' created successfully.


In [5]:
# ---------------------------------------------------------------
# Create "folders" inside the bucket
# ---------------------------------------------------------------
# S3 has no real folders - they are just zero-byte objects whose keys
# end in "/". Creating them mirrors clicking "Create folder" 3 times.
folders = [KENDRA_PREFIX, PERSONALIZE_PREFIX, A2I_OUTPUT_PREFIX]

for folder in folders:
    s3_client.put_object(Bucket=BUCKET_NAME, Key=folder)
    print(f"Created folder: s3://{BUCKET_NAME}/{folder}")


Created folder: s3://aws-ai-services-745404749079-us-east-1-49f4e565/kendra-docs/


Created folder: s3://aws-ai-services-745404749079-us-east-1-49f4e565/personalize-data/


Created folder: s3://aws-ai-services-745404749079-us-east-1-49f4e565/a2i-output/


### Upload Lab Documents

The original lab requires two input files (the same ones linked in the guide's *Prerequisite* section):

1. **A sample document for Kendra** (e.g. `return-policy.txt`) — any short text file describing a return policy or FAQ.
2. **`interactions.csv`** — customer interaction data with columns `USER_ID, ITEM_ID, TIMESTAMP, EVENT_TYPE` (minimum 1000 rows).

In the console lab, these were downloaded from Google Drive links and manually uploaded. In this notebook:

- **If you've already uploaded `return-policy.txt` and/or `interactions.csv` using Jupyter's Upload button**, the cells below automatically detect them in your current notebook's working directory — no path editing needed. Just make sure the uploaded files sit in the **same folder as this `.ipynb` file** (that's where Jupyter's Upload button places them by default).
- **If no matching file is found**, the cell automatically falls back to generating a sample `return-policy.txt` and a synthetic 1200-row `interactions.csv`, so the notebook still runs end-to-end without any manual downloads.

> **Tip:** If your uploaded file has a different name than `return-policy.txt` or `interactions.csv`, either rename it to match, or add your filename to the `CANDIDATE_FILENAMES` / `CANDIDATE_CSV_FILENAMES` list in the next two cells.


In [6]:
# ---------------------------------------------------------------
# Locate your uploaded Kendra document, or generate a sample one
# ---------------------------------------------------------------
# IMPORTANT: SageMaker Notebook instances use a different filesystem than
# this notebook's authoring environment. The cell below auto-detects your
# current Jupyter working directory and looks for a file you uploaded
# there, so you do NOT need to hardcode a path.
import os

# Your current Jupyter working directory (wherever this .ipynb file lives,
# and where the Jupyter "Upload" button places files by default)
NOTEBOOK_DIR = os.getcwd()
print(f"Current working directory: {NOTEBOOK_DIR}")

# Common filenames learners use for the Kendra sample document -
# edit this list if your uploaded file has a different name
CANDIDATE_FILENAMES = ["return-policy.txt", "return_policy.txt", "returnpolicy.txt"]

local_kendra_doc = None
for fname in CANDIDATE_FILENAMES:
    candidate_path = os.path.join(NOTEBOOK_DIR, fname)
    if os.path.exists(candidate_path):
        local_kendra_doc = candidate_path
        break

if local_kendra_doc:
    print(f"Found your uploaded file: {local_kendra_doc}")
else:
    # Fallback: no uploaded file found, so generate a sample one
    # so the notebook still runs end-to-end without manual downloads
    return_policy_text = '''Return Policy - Retail Company FAQ

Q: What is the return policy?
A: Items can be returned within 30 days of purchase with a valid receipt.
Refunds are issued to the original payment method within 5-7 business days.

Q: How do customers request refunds?
A: Customers can request a refund by visiting the Returns section of our
website, or by bringing the item and receipt to any physical store location.

Q: Are there items that cannot be returned?
A: Final sale items, gift cards, and personal care products cannot be returned
unless defective.

Q: What if I do not have a receipt?
A: A valid form of ID can be used to look up the purchase in our system.
'''
    local_kendra_doc = os.path.join(NOTEBOOK_DIR, "return-policy.txt")
    with open(local_kendra_doc, "w") as f:
        f.write(return_policy_text)
    print(f"No uploaded file found in {NOTEBOOK_DIR} - generated a sample instead: {local_kendra_doc}")

print(f"\nFinal path being used: {local_kendra_doc}")


Current working directory: /home/sagemaker-user
Found your uploaded file: /home/sagemaker-user/return-policy.txt

Final path being used: /home/sagemaker-user/return-policy.txt


In [7]:
# ---------------------------------------------------------------
# Locate your uploaded interactions.csv, or generate a synthetic one
# ---------------------------------------------------------------
import random
import csv

CANDIDATE_CSV_FILENAMES = ["interactions.csv", "interactions_csv.csv", "interaction.csv"]

local_interactions_csv = None
for fname in CANDIDATE_CSV_FILENAMES:
    candidate_path = os.path.join(NOTEBOOK_DIR, fname)
    if os.path.exists(candidate_path):
        local_interactions_csv = candidate_path
        break

if local_interactions_csv:
    print(f"Found your uploaded file: {local_interactions_csv}")
else:
    # Fallback: generate a synthetic but valid interactions.csv (1000+ rows)
    # Columns required by Personalize's default Interactions schema:
    # USER_ID, ITEM_ID, TIMESTAMP, EVENT_TYPE
    local_interactions_csv = os.path.join(NOTEBOOK_DIR, "interactions.csv")

    NUM_USERS = 50
    NUM_ITEMS = 100
    NUM_ROWS = 1200  # safely above the 1000-row minimum mentioned in the guide
    EVENT_TYPES = ["view", "click", "purchase"]

    # Start timestamp ~90 days ago, in Unix epoch seconds (Personalize default format)
    start_ts = int(time.time()) - (90 * 24 * 60 * 60)

    with open(local_interactions_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["USER_ID", "ITEM_ID", "TIMESTAMP", "EVENT_TYPE"])
        for _ in range(NUM_ROWS):
            user_id = random.randint(1, NUM_USERS)
            item_id = random.randint(1, NUM_ITEMS)
            ts = start_ts + random.randint(0, 90 * 24 * 60 * 60)
            event = random.choices(EVENT_TYPES, weights=[0.6, 0.3, 0.1])[0]
            writer.writerow([user_id, item_id, ts, event])

    print(f"No uploaded file found in {NOTEBOOK_DIR} - generated a synthetic dataset instead.")
    print(f"NOTE: If your lab requires a SPECIFIC dataset (e.g. graded on exact content),")
    print(f"upload your own interactions.csv into the same folder as this notebook and re-run this cell.")

print(f"\nFinal path being used: {local_interactions_csv}")


Found your uploaded file: /home/sagemaker-user/interactions.csv

Final path being used: /home/sagemaker-user/interactions.csv


In [8]:
# ---------------------------------------------------------------
# Upload both files to S3 (equivalent to the "Upload" button clicks)
# ---------------------------------------------------------------
kendra_doc_key = f"{KENDRA_PREFIX}return-policy.txt"
interactions_key = f"{PERSONALIZE_PREFIX}interactions.csv"

s3_client.upload_file(local_kendra_doc, BUCKET_NAME, kendra_doc_key)
print(f"Uploaded: s3://{BUCKET_NAME}/{kendra_doc_key}")

s3_client.upload_file(local_interactions_csv, BUCKET_NAME, interactions_key)
print(f"Uploaded: s3://{BUCKET_NAME}/{interactions_key}")

# Build the full S3 URI Personalize will need later
INTERACTIONS_S3_URI = f"s3://{BUCKET_NAME}/{interactions_key}"
print(f"\nInteractions S3 URI (save for later): {INTERACTIONS_S3_URI}")


Uploaded: s3://aws-ai-services-745404749079-us-east-1-49f4e565/kendra-docs/return-policy.txt


Uploaded: s3://aws-ai-services-745404749079-us-east-1-49f4e565/personalize-data/interactions.csv

Interactions S3 URI (save for later): s3://aws-ai-services-745404749079-us-east-1-49f4e565/personalize-data/interactions.csv


### Configure the Bucket Policy

This replaces editing the **Permissions → Bucket policy** section in the console. The policy below grants `personalize.amazonaws.com` (and `kendra.amazonaws.com`, needed later) read access to the bucket — equivalent to the JSON policy pasted in the original guide, but scoped correctly to your dynamically-created bucket name (no manual find-and-replace needed).


In [9]:
# ---------------------------------------------------------------
# Attach a bucket policy granting AWS AI services read access
# ---------------------------------------------------------------
bucket_policy = {
    "Version": "2012-10-17",
    "Id": "AIServicesS3BucketAccessPolicy",
    "Statement": [
        {
            "Sid": "PersonalizeS3BucketAccessPolicy",
            "Effect": "Allow",
            "Principal": {"Service": "personalize.amazonaws.com"},
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*"
            ]
        },
        {
            "Sid": "KendraS3BucketAccessPolicy",
            "Effect": "Allow",
            "Principal": {"Service": "kendra.amazonaws.com"},
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*"
            ]
        }
    ]
}

s3_client.put_bucket_policy(
    Bucket=BUCKET_NAME,
    Policy=json.dumps(bucket_policy)
)
print("Bucket policy applied successfully.")
print(json.dumps(bucket_policy, indent=2))


Bucket policy applied successfully.
{
  "Version": "2012-10-17",
  "Id": "AIServicesS3BucketAccessPolicy",
  "Statement": [
    {
      "Sid": "PersonalizeS3BucketAccessPolicy",
      "Effect": "Allow",
      "Principal": {
        "Service": "personalize.amazonaws.com"
      },
      "Action": [
        "s3:GetObject",
        "s3:ListBucket"
      ],
      "Resource": [
        "arn:aws:s3:::aws-ai-services-745404749079-us-east-1-49f4e565",
        "arn:aws:s3:::aws-ai-services-745404749079-us-east-1-49f4e565/*"
      ]
    },
    {
      "Sid": "KendraS3BucketAccessPolicy",
      "Effect": "Allow",
      "Principal": {
        "Service": "kendra.amazonaws.com"
      },
      "Action": [
        "s3:GetObject",
        "s3:ListBucket"
      ],
      "Resource": [
        "arn:aws:s3:::aws-ai-services-745404749079-us-east-1-49f4e565",
        "arn:aws:s3:::aws-ai-services-745404749079-us-east-1-49f4e565/*"
      ]
    }
  ]
}


---
## Section 2: Create IAM Roles

**Console steps being replaced:**
- "Create a new role (Recommended)" while creating the Kendra index
- Attaching `AmazonS3FullAccess` to the Kendra role
- "Create a new role" while creating the Personalize import job
- Attaching `AmazonS3FullAccess` + a custom inline policy to the Personalize role
- "Create a new role" while creating the A2I workflow

Instead of clicking through the IAM console for each service, we define one helper function that creates a role with a trust policy for a given AWS service, then attach permissions to it. This is the programmatic equivalent of IAM's "Create a new role (Recommended)" option, which does exactly this under the hood.

> **Note:** In production you would scope these policies far more narrowly than `AmazonS3FullAccess`. We use it here only because the **original console guide itself uses `AmazonS3FullAccess`** for simplicity — to keep this notebook a faithful 1:1 equivalent of the original lab.


In [10]:
# ---------------------------------------------------------------
# Helper function: create an IAM role with a trust policy for a
# given AWS service, then attach a managed policy to it.
# ---------------------------------------------------------------
def create_service_role(role_name, service_principal, managed_policy_arns, inline_policies=None):
    """
    Creates (or reuses) an IAM role that a given AWS service can assume.

    role_name           : name for the new IAM role
    service_principal    : e.g. 'kendra.amazonaws.com'
    managed_policy_arns  : list of AWS managed policy ARNs to attach
    inline_policies       : optional dict of {policy_name: policy_document}
    """
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": service_principal},
                "Action": "sts:AssumeRole"
            }
        ]
    }

    try:
        response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"Role allowing {service_principal} to access AWS resources for this lab"
        )
        role_arn = response["Role"]["Arn"]
        print(f"Created role: {role_name}")
    except iam_client.exceptions.EntityAlreadyExistsException:
        role_arn = iam_client.get_role(RoleName=role_name)["Role"]["Arn"]
        print(f"Role already exists, reusing: {role_name}")

    for policy_arn in managed_policy_arns:
        iam_client.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)
        print(f"  Attached managed policy: {policy_arn}")

    if inline_policies:
        for policy_name, policy_doc in inline_policies.items():
            iam_client.put_role_policy(
                RoleName=role_name,
                PolicyName=policy_name,
                PolicyDocument=json.dumps(policy_doc)
            )
            print(f"  Attached inline policy: {policy_name}")

    # IAM roles take a few seconds to propagate across AWS - this avoids
    # "role cannot be assumed" errors on the very next API call
    time.sleep(10)
    return role_arn


In [11]:
# ---------------------------------------------------------------
# Role 1: Amazon Kendra role
# ---------------------------------------------------------------
# Kendra needs THREE permission sets:
#   1. AmazonS3FullAccess          — read documents from S3
#   2. CloudwatchLogsFullAccess    — create/write sync job log groups (REQUIRED)
#   3. AmazonKendraFullAccess      — manage the index itself
#
# The original notebook only attached S3 access, which causes:
#   "Failed to create LogGroup ... insufficient permissions"
# during the data source sync job.

kendra_cloudwatch_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "KendraCloudWatchLogsAccess",
            "Effect": "Allow",
            "Action": [
                "cloudwatch:PutMetricData",
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:DescribeLogGroups",
                "logs:DescribeLogStreams",
                "logs:PutLogEvents"
            ],
            "Resource": "*"
        }
    ]
}

kendra_role_arn = create_service_role(
    role_name=f"AmazonKendra-{REGION}-role-{UNIQUE_SUFFIX}",
    service_principal="kendra.amazonaws.com",
    managed_policy_arns=[
        "arn:aws:iam::aws:policy/AmazonS3FullAccess",
        "arn:aws:iam::aws:policy/CloudWatchLogsFullAccess",
        "arn:aws:iam::aws:policy/AmazonKendraFullAccess"
    ],
    inline_policies={"KendraCloudWatchLogsPolicy": kendra_cloudwatch_policy}
)
print(f"\nKendra Role ARN: {kendra_role_arn}")


Created role: AmazonKendra-us-east-1-role-49f4e565


  Attached managed policy: arn:aws:iam::aws:policy/AmazonS3FullAccess


  Attached managed policy: arn:aws:iam::aws:policy/CloudWatchLogsFullAccess


  Attached managed policy: arn:aws:iam::aws:policy/AmazonKendraFullAccess


  Attached inline policy: KendraCloudWatchLogsPolicy



Kendra Role ARN: arn:aws:iam::745404749079:role/AmazonKendra-us-east-1-role-49f4e565


In [12]:
# ---------------------------------------------------------------
# FIX: If your Kendra role already exists but is missing CloudWatch
# permissions (causing sync FAILED), run this cell to patch it.
# Safe to run even if the role was just created above.
# ---------------------------------------------------------------
KENDRA_ROLE_NAME = f"AmazonKendra-{REGION}-role-{UNIQUE_SUFFIX}"

cloudwatch_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "KendraCloudWatchLogsAccess",
            "Effect": "Allow",
            "Action": [
                "cloudwatch:PutMetricData",
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:DescribeLogGroups",
                "logs:DescribeLogStreams",
                "logs:PutLogEvents"
            ],
            "Resource": "*"
        }
    ]
}

# Attach CloudWatchLogsFullAccess managed policy
try:
    iam_client.attach_role_policy(
        RoleName=KENDRA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/CloudWatchLogsFullAccess"
    )
    print("Attached CloudWatchLogsFullAccess to Kendra role.")
except Exception as e:
    print(f"Managed policy note: {e}")

# Attach AmazonKendraFullAccess managed policy
try:
    iam_client.attach_role_policy(
        RoleName=KENDRA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/AmazonKendraFullAccess"
    )
    print("Attached AmazonKendraFullAccess to Kendra role.")
except Exception as e:
    print(f"Managed policy note: {e}")

# Add granular inline policy for CloudWatch Logs
iam_client.put_role_policy(
    RoleName=KENDRA_ROLE_NAME,
    PolicyName="KendraCloudWatchLogsPolicy",
    PolicyDocument=json.dumps(cloudwatch_inline)
)
print("Inline CloudWatch Logs policy applied.")

# IAM propagation delay
print("Waiting 15s for IAM changes to propagate...")
time.sleep(15)
print("Done. Now re-run the Kendra sync cell.")


Attached CloudWatchLogsFullAccess to Kendra role.


Attached AmazonKendraFullAccess to Kendra role.


Inline CloudWatch Logs policy applied.
Waiting 15s for IAM changes to propagate...


Done. Now re-run the Kendra sync cell.


In [13]:
# ---------------------------------------------------------------
# Role 2: Amazon Personalize role
# ---------------------------------------------------------------
# Equivalent to the inline policy named "AI-Services-Execution-Role"
# from the original guide, scoped to our bucket.
personalize_inline_policy = {
    "Version": "2012-10-17",
    "Id": "PersonalizeS3BucketAccessPolicy",
    "Statement": [
        {
            "Sid": "PersonalizeS3BucketAccessPolicy",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*"
            ]
        }
    ]
}

personalize_role_arn = create_service_role(
    role_name=f"AmazonPersonalize-role-{UNIQUE_SUFFIX}",
    service_principal="personalize.amazonaws.com",
    managed_policy_arns=["arn:aws:iam::aws:policy/AmazonS3FullAccess"],
    inline_policies={"AI-Services-Execution-Role": personalize_inline_policy}
)
print(f"\nPersonalize Role ARN: {personalize_role_arn}")


Created role: AmazonPersonalize-role-49f4e565


  Attached managed policy: arn:aws:iam::aws:policy/AmazonS3FullAccess


  Attached inline policy: AI-Services-Execution-Role



Personalize Role ARN: arn:aws:iam::745404749079:role/AmazonPersonalize-role-49f4e565


In [14]:
# ---------------------------------------------------------------
# Role 3: Amazon A2I (SageMaker) role
# ---------------------------------------------------------------
a2i_role_arn = create_service_role(
    role_name=f"AmazonA2I-role-{UNIQUE_SUFFIX}",
    service_principal="sagemaker.amazonaws.com",
    managed_policy_arns=[
        "arn:aws:iam::aws:policy/AmazonS3FullAccess",
        "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess"
    ]
)
print(f"\nA2I (SageMaker) Role ARN: {a2i_role_arn}")


Created role: AmazonA2I-role-49f4e565


  Attached managed policy: arn:aws:iam::aws:policy/AmazonS3FullAccess


  Attached managed policy: arn:aws:iam::aws:policy/AmazonSageMakerFullAccess



A2I (SageMaker) Role ARN: arn:aws:iam::745404749079:role/AmazonA2I-role-49f4e565


---
## Section 3: Create an Amazon Kendra Index

**Console steps being replaced:**
- "Create an index" → name it, select Developer edition, choose IAM role
- "Add data source" → Amazon S3 connector
- Configure data source details, IAM role, S3 location, include pattern `kendra-docs/`
- "Sync now"
- "Search indexed content" to test queries

> ⚠️ **Billing reminder (same as the original guide):** Kendra Developer Edition bills **per hour** the index exists, regardless of usage. Don't forget the Cleanup section at the end.


In [15]:
# ---------------------------------------------------------------
# Create the Kendra index
# ---------------------------------------------------------------
kendra_response = kendra_client.create_index(
    Name=f"business-search-index-{UNIQUE_SUFFIX}",
    Edition="DEVELOPER_EDITION",   # matches the console's "Developer edition" choice
    RoleArn=kendra_role_arn,
    Description="Intelligent document search index for the retail Business Assistant lab"
)
KENDRA_INDEX_ID = kendra_response["Id"]
print(f"Kendra index creation started. Index ID: {KENDRA_INDEX_ID}")
print("This typically takes 10-30 minutes to become ACTIVE.")


Kendra index creation started. Index ID: 6be3954d-1296-4e4b-86fe-7ae4799dd695
This typically takes 10-30 minutes to become ACTIVE.


In [16]:
# ---------------------------------------------------------------
# Wait for the index to become ACTIVE before continuing
# ---------------------------------------------------------------
# Equivalent to watching the console status change from "Creating" to "Active"
def wait_for_kendra_index(index_id, poll_seconds=30):
    while True:
        status = kendra_client.describe_index(Id=index_id)["Status"]
        print(f"  Kendra index status: {status}")
        if status == "ACTIVE":
            print("Kendra index is ACTIVE.")
            break
        elif status == "FAILED":
            raise Exception("Kendra index creation FAILED. Check the console for details.")
        time.sleep(poll_seconds)

wait_for_kendra_index(KENDRA_INDEX_ID)


  Kendra index status: CREATING


  Kendra index status: ACTIVE
Kendra index is ACTIVE.


In [17]:
# ---------------------------------------------------------------
# Add an S3 data source to the Kendra index
# ---------------------------------------------------------------
datasource_response = kendra_client.create_data_source(
    IndexId=KENDRA_INDEX_ID,
    Name=f"company-document-{UNIQUE_SUFFIX}",
    Type="S3",
    Configuration={
        "S3Configuration": {
            "BucketName": BUCKET_NAME,
            "InclusionPrefixes": [KENDRA_PREFIX]
        }
    },
    RoleArn=kendra_role_arn,
    Description="S3 data source for company documents (return policy, FAQs)"
)
KENDRA_DATASOURCE_ID = datasource_response["Id"]
print(f"Kendra data source created. Data source ID: {KENDRA_DATASOURCE_ID}")

# ---------------------------------------------------------------
# Wait for the data source to reach ACTIVE before syncing
# ---------------------------------------------------------------
# Kendra data sources go through CREATING -> ACTIVE.
# Calling start_data_source_sync_job while still CREATING raises:
#   ConflictException: Connector is not in ready state to run.
print("Waiting for data source to become ACTIVE...")
import time
while True:
    ds_status = kendra_client.describe_data_source(
        Id=KENDRA_DATASOURCE_ID,
        IndexId=KENDRA_INDEX_ID
    )["Status"]
    print(f"  Data source status: {ds_status}")
    if ds_status == "ACTIVE":
        print("Data source is ACTIVE — ready to sync.")
        break
    elif ds_status == "FAILED":
        raise Exception("Data source creation FAILED. Check IAM role permissions.")
    time.sleep(10)


Kendra data source created. Data source ID: 536b0593-c8b5-47e4-ae8f-63b184c5ba3b
Waiting for data source to become ACTIVE...


  Data source status: CREATING


  Data source status: ACTIVE
Data source is ACTIVE — ready to sync.


In [18]:
# ---------------------------------------------------------------
# Start the sync job (equivalent to clicking "Sync now")
# ---------------------------------------------------------------
# Retries handle any brief propagation delay between ACTIVE status
# being reported and the sync API accepting the request.
import time

MAX_RETRIES = 5
for attempt in range(1, MAX_RETRIES + 1):
    try:
        sync_response = kendra_client.start_data_source_sync_job(
            Id=KENDRA_DATASOURCE_ID,
            IndexId=KENDRA_INDEX_ID
        )
        print(f"Sync job started: {sync_response['ExecutionId']}")
        break
    except kendra_client.exceptions.ConflictException as e:
        if attempt < MAX_RETRIES:
            print(f"  Attempt {attempt}: data source not ready yet, retrying in 15s... ({e})")
            time.sleep(15)
        else:
            raise Exception(
                f"Sync job failed to start after {MAX_RETRIES} attempts.\n"
                f"Check data source status in the Kendra console.\n"
                f"Error: {e}"
            )


Sync job started: f7040d30-b42a-4a89-aa13-497c777a132a


In [19]:
# ---------------------------------------------------------------
# Poll sync status (equivalent to watching "Sync history" in console)
# ---------------------------------------------------------------
def wait_for_kendra_sync(index_id, datasource_id, poll_seconds=30):
    while True:
        history = kendra_client.list_data_source_sync_jobs(
            Id=datasource_id, IndexId=index_id
        )
        jobs = history.get("History", [])
        if jobs:
            latest = jobs[0]
            status = latest["Status"]
            print(f"  Sync status: {status}")
            if status == "SUCCEEDED":
                print("Sync completed successfully.")
                break
            elif status == "FAILED":
                print("Sync FAILED. Details:", latest.get("ErrorMessage", "N/A"))
                break
        time.sleep(poll_seconds)

wait_for_kendra_sync(KENDRA_INDEX_ID, KENDRA_DATASOURCE_ID)


  Sync status: SYNCING


  Sync status: SUCCEEDED
Sync completed successfully.


### Test the Search — Natural Language Queries

This replaces the **"Search indexed content"** console page where you typed queries like *"What is the return policy?"*.


In [20]:
# ---------------------------------------------------------------
# Query the Kendra index with natural language questions
# ---------------------------------------------------------------
def search_kendra(query_text):
    response = kendra_client.query(
        IndexId=KENDRA_INDEX_ID,
        QueryText=query_text
    )
    print(f"\nQuery: \"{query_text}\"")
    if not response["ResultItems"]:
        print("  No results found. (If the sync just finished, indexing may need a minute more.)")
    for item in response["ResultItems"][:3]:
        excerpt = item.get("DocumentExcerpt", {}).get("Text", "")
        print(f"  -> {excerpt[:300]}")

search_kendra("What is the return policy?")
search_kendra("How do customers request refunds?")



Query: "What is the return policy?"
  -> Return Policy - Retail Company FAQ

Q: What is the return policy?
A: Items can be returned within 30 days of purchase with a valid receipt.
Refunds are issued to the original payment method within 5-7 business days.

Q: How do customers request refunds?
A: Customers can request a refund by visiting 
  -> ...Return Policy - Retail Company FAQ

Q: What is the return policy?
A: Items can be returned within 30 days of purchase with a valid receipt.
Refunds are issued to the original payment method...



Query: "How do customers request refunds?"
  -> Return Policy - Retail Company FAQ

Q: What is the return policy?
A: Items can be returned within 30 days of purchase with a valid receipt.
Refunds are issued to the original payment method within 5-7 business days.

Q: How do customers request refunds?
A: Customers can request a refund by visiting 
  -> ...A: Items can be returned within 30 days of purchase with a valid receipt.
Refunds are issued to the original payment method within 5-7 business days.

Q: How do customers request refunds?
A: Customers can request a refund by visiting the Returns section of our
website, or by bringing the item and


---
## Section 4: Create an Amazon Personalize Dataset Group

**Console steps being replaced:**
- "Create dataset group" → name, E-commerce domain
- Create dataset → "Create a new domain schema"
- "Import data from S3" → dataset import job
- "Create recommenders" → "Recommended for you" use case
- "Test" recommender with a User ID


In [21]:
# ---------------------------------------------------------------
# Create the Personalize dataset group (E-commerce domain)
# ---------------------------------------------------------------
dataset_group_response = personalize_client.create_dataset_group(
    name=f"retail-recommendation-group-{UNIQUE_SUFFIX}",
    domain="ECOMMERCE"   # matches the console's "E-commerce domain" selection
)
DATASET_GROUP_ARN = dataset_group_response["datasetGroupArn"]
print(f"Dataset group ARN: {DATASET_GROUP_ARN}")


Dataset group ARN: arn:aws:personalize:us-east-1:745404749079:dataset-group/retail-recommendation-group-49f4e565


In [22]:
# ---------------------------------------------------------------
# Wait for the dataset group to become ACTIVE
# ---------------------------------------------------------------
def wait_for_personalize_resource(describe_fn, arn_key, arn_value, poll_seconds=20):
    """Generic poller for any Personalize resource's 'status' field."""
    while True:
        response = describe_fn(**{arn_key: arn_value})
        # response structure differs slightly per resource type, so we grab
        # the first nested dict's 'status' field generically
        inner_key = [k for k in response.keys() if k != "ResponseMetadata"][0]
        status = response[inner_key]["status"]
        print(f"  Status: {status}")
        if status == "ACTIVE":
            print("Resource is ACTIVE.")
            break
        elif "FAILED" in status:
            raise Exception(f"Resource creation FAILED: {response[inner_key]}")
        time.sleep(poll_seconds)

wait_for_personalize_resource(
    personalize_client.describe_dataset_group, "datasetGroupArn", DATASET_GROUP_ARN
)


  Status: CREATE PENDING


  Status: CREATE PENDING


  Status: ACTIVE
Resource is ACTIVE.


In [23]:
# ---------------------------------------------------------------
# Get the default "Interactions" schema for the E-commerce domain
# ---------------------------------------------------------------
# Equivalent to: "Create a new domain schema by modifying the existing
# default schema for your domain"
default_schemas = personalize_client.list_schemas()
ecommerce_interactions_schema = None

# Personalize provides built-in domain schemas we can reference directly;
# we create our own copy named to match the original guide
interactions_schema_definition = {
    "type": "record",
    "name": "Interactions",
    "namespace": "com.amazonaws.personalize.schema",
    "fields": [
        {"name": "USER_ID", "type": "string"},
        {"name": "ITEM_ID", "type": "string"},
        {"name": "TIMESTAMP", "type": "long"},
        {"name": "EVENT_TYPE", "type": "string"}
    ],
    "version": "1.0"
}

schema_response = personalize_client.create_schema(
    name=f"retail-interactions-schema-{UNIQUE_SUFFIX}",
    schema=json.dumps(interactions_schema_definition),
    domain="ECOMMERCE"
)
INTERACTIONS_SCHEMA_ARN = schema_response["schemaArn"]
print(f"Interactions schema ARN: {INTERACTIONS_SCHEMA_ARN}")


Interactions schema ARN: arn:aws:personalize:us-east-1:745404749079:schema/retail-interactions-schema-49f4e565


In [24]:
# ---------------------------------------------------------------
# Create the Interactions dataset inside the dataset group
# ---------------------------------------------------------------
dataset_response = personalize_client.create_dataset(
    name=f"retail-interactions-dataset-{UNIQUE_SUFFIX}",
    schemaArn=INTERACTIONS_SCHEMA_ARN,
    datasetGroupArn=DATASET_GROUP_ARN,
    datasetType="INTERACTIONS"
)
INTERACTIONS_DATASET_ARN = dataset_response["datasetArn"]
print(f"Interactions dataset ARN: {INTERACTIONS_DATASET_ARN}")


Interactions dataset ARN: arn:aws:personalize:us-east-1:745404749079:dataset/retail-recommendation-group-49f4e565/INTERACTIONS


In [26]:
# ---------------------------------------------------------------
# Import interactions.csv from S3 into the dataset
# ---------------------------------------------------------------
# Equivalent to: "Import data from S3" -> dataset import job -> Data location
import_job_response = personalize_client.create_dataset_import_job(
    jobName=f"my-dataset-import-{UNIQUE_SUFFIX}",
    datasetArn=INTERACTIONS_DATASET_ARN,
    dataSource={"dataLocation": INTERACTIONS_S3_URI},
    roleArn=personalize_role_arn
)
IMPORT_JOB_ARN = import_job_response["datasetImportJobArn"]
print(f"Dataset import job started: {IMPORT_JOB_ARN}")


Dataset import job started: arn:aws:personalize:us-east-1:745404749079:dataset-import-job/my-dataset-import-49f4e565


In [27]:
# ---------------------------------------------------------------
# Poll until the dataset import job is ACTIVE
# ---------------------------------------------------------------
# This corresponds to watching the status change:
# "Item interactions dataset is being uploaded" -> "...active"
def wait_for_import_job(job_arn, poll_seconds=30):
    while True:
        status = personalize_client.describe_dataset_import_job(
            datasetImportJobArn=job_arn
        )["datasetImportJob"]["status"]
        print(f"  Import job status: {status}")
        if status == "ACTIVE":
            print("Dataset import completed successfully.")
            break
        elif "FAILED" in status:
            raise Exception("Dataset import job FAILED. Check CloudWatch logs for details.")
        time.sleep(poll_seconds)

wait_for_import_job(IMPORT_JOB_ARN)


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: CREATE IN_PROGRESS


  Import job status: ACTIVE
Dataset import completed successfully.


### Create the Recommender ("Recommended for you")

This replaces: **"Set up training and recommendation resources" → "Use e-commerce recommenders" → "Recommended for you" use case → "Create recommenders"**.

> **Note on training time:** Just like in the console (where the status goes `Create pending` → `Active`), training a Personalize recommender can take **30 minutes to a few hours** depending on data size. The polling cell below will block until training completes — feel free to reduce `poll_seconds` or run it in the background if you want to keep working in other cells.


In [28]:
# ---------------------------------------------------------------
# Create the "Recommended for you" recommender
# ---------------------------------------------------------------
# This is the Personalize "Recipe ARN" corresponding to the
# E-commerce domain's "Recommended for you" use case
RECOMMENDED_FOR_YOU_RECIPE_ARN = "arn:aws:personalize:::recipe/aws-ecomm-recommended-for-you"

recommender_response = personalize_client.create_recommender(
    name=f"product-recommender-{UNIQUE_SUFFIX}",
    recipeArn=RECOMMENDED_FOR_YOU_RECIPE_ARN,
    datasetGroupArn=DATASET_GROUP_ARN
)
RECOMMENDER_ARN = recommender_response["recommenderArn"]
print(f"Recommender creation started: {RECOMMENDER_ARN}")
print("Status will move: CREATE PENDING -> CREATE IN_PROGRESS -> ACTIVE")


Recommender creation started: arn:aws:personalize:us-east-1:745404749079:recommender/product-recommender-49f4e565
Status will move: CREATE PENDING -> CREATE IN_PROGRESS -> ACTIVE


In [29]:
# ---------------------------------------------------------------
# Poll recommender status until ACTIVE
# ---------------------------------------------------------------
def wait_for_recommender(recommender_arn, poll_seconds=60):
    while True:
        status = personalize_client.describe_recommender(
            recommenderArn=recommender_arn
        )["recommender"]["status"]
        print(f"  Recommender status: {status}")
        if status == "ACTIVE":
            print("Recommender is ACTIVE and ready to serve recommendations.")
            break
        elif "FAILED" in status:
            raise Exception("Recommender training FAILED. Check the console for failure reason.")
        time.sleep(poll_seconds)

wait_for_recommender(RECOMMENDER_ARN)


  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


...

  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


  Recommender status: CREATE IN_PROGRESS


  Recommender status: ACTIVE
Recommender is ACTIVE and ready to serve recommendations.


### Test Recommendations — Equivalent to the Console's "Test" Button

The original guide enters a User ID (e.g. `1168`) and clicks **"Get recommendations"**. We do the same thing here with `get_recommendations()` from the Personalize Runtime client.


In [30]:
# ---------------------------------------------------------------
# Get personalized recommendations for a test user
# ---------------------------------------------------------------
TEST_USER_ID = "5"  # pick any USER_ID that exists in your interactions.csv

rec_response = personalize_runtime_client.get_recommendations(
    recommenderArn=RECOMMENDER_ARN,
    userId=TEST_USER_ID,
    numResults=10
)

print(f"Recommendations for User ID {TEST_USER_ID}:\n")
for item in rec_response["itemList"]:
    print(f"  Item ID: {item['itemId']:>5}   Score: {item.get('score', 'N/A')}")


Recommendations for User ID 5:

  Item ID:     9   Score: 0.0209202
  Item ID:    73   Score: 0.0188998
  Item ID:    16   Score: 0.0184653
  Item ID:    61   Score: 0.0176936
  Item ID:    13   Score: 0.0176573
  Item ID:    82   Score: 0.0172201
  Item ID:    31   Score: 0.0167844
  Item ID:    72   Score: 0.0160694
  Item ID:    35   Score: 0.0148734
  Item ID:    97   Score: 0.0143571


---
## Section 5: Create an Amazon A2I Human Review Workflow

**Console steps being replaced:**
- "Create human review workflow" → name, S3 output path, IAM role
- "Custom" task type
- "Create a new custom template" (the `crowd-classifier` HTML template)
- "Private" worker type → "Create a new team" (Cognito-backed private workforce)
- Adding a reviewer's email + organization details
- Verifying the workflow is `Active`

> **Important limitation:** Creating the **private workforce team itself** (the Cognito User Pool, sending the reviewer an email invite, them setting a password, etc.) is a multi-party, identity-verification process that **cannot be fully automated via a single boto3 call** — AWS deliberately keeps workforce *onboarding* (the human accepting an invite) outside of pure API automation for security reasons. Below, we automate everything we can — creating the **private workforce** and the **work team** via the SageMaker API — and clearly flag the one manual step that remains (the reviewer accepting their email invite), exactly as a real engineer integrating this into infrastructure-as-code would do.


In [31]:
# ---------------------------------------------------------------
# Step 1: Auto-provision Cognito User Pool + Create Private Workforce
# ---------------------------------------------------------------
# This cell fully automates what the console does "behind the scenes":
# 1. Creates a Cognito User Pool
# 2. Creates an App Client inside it
# 3. Creates a User Group
# 4. Creates the SageMaker private workforce using those IDs
# If a private workforce already exists in this region, it reuses it.

import boto3

REVIEWER_EMAIL    = "reviewer@example.com"   # <-- CHANGE to a real email for lab grading
ORGANIZATION_NAME = "k21academy"
CONTACT_EMAIL     = "contact@example.com"    # <-- CHANGE to a real email

cognito_client = session.client("cognito-idp")

WORKFORCE_ARN      = None
COGNITO_POOL_ID    = None
COGNITO_CLIENT_ID  = None
COGNITO_GROUP_NAME = f"review-group-{UNIQUE_SUFFIX}"

# ── Check if a private workforce already exists (1 per region limit) ────────
existing_workforces = sagemaker_client.list_workforces()["Workforces"]

if existing_workforces:
    WORKFORCE_ARN = existing_workforces[0]["WorkforceArn"]
    print(f"Found existing private workforce — reusing: {WORKFORCE_ARN}")
    # Try to pull Cognito IDs from the existing workforce
    wf_details = sagemaker_client.describe_workforce(
        WorkforceName=existing_workforces[0]["WorkforceName"]
    )["Workforce"]
    cognito_cfg = wf_details.get("CognitoConfig", {})
    COGNITO_POOL_ID   = cognito_cfg.get("UserPool", None)
    COGNITO_CLIENT_ID = cognito_cfg.get("ClientId", None)
    print(f"  Cognito Pool ID : {COGNITO_POOL_ID}")
    print(f"  Cognito Client ID: {COGNITO_CLIENT_ID}")
else:
    # ── Create Cognito User Pool ────────────────────────────────────────────
    pool_response = cognito_client.create_user_pool(
        PoolName=f"a2i-workforce-pool-{UNIQUE_SUFFIX}",
        AutoVerifiedAttributes=["email"],
        UsernameAttributes=["email"],
        Policies={
            "PasswordPolicy": {
                "MinimumLength": 8,
                "RequireUppercase": True,
                "RequireLowercase": True,
                "RequireNumbers": True,
                "RequireSymbols": False
            }
        }
    )
    COGNITO_POOL_ID = pool_response["UserPool"]["Id"]
    print(f"Cognito User Pool created: {COGNITO_POOL_ID}")

    # ── Create App Client (no secret, required by SageMaker) ───────────────
    client_response = cognito_client.create_user_pool_client(
        UserPoolId=COGNITO_POOL_ID,
        ClientName=f"a2i-app-client-{UNIQUE_SUFFIX}",
        GenerateSecret=False,   # SageMaker A2I requires no client secret
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"]
    )
    COGNITO_CLIENT_ID = client_response["UserPoolClient"]["ClientId"]
    print(f"Cognito App Client created: {COGNITO_CLIENT_ID}")

    # ── Create User Group ────────────────────────────────────────────────────
    cognito_client.create_group(
        UserPoolId=COGNITO_POOL_ID,
        GroupName=COGNITO_GROUP_NAME,
        Description="A2I reviewer group"
    )
    print(f"Cognito User Group created: {COGNITO_GROUP_NAME}")

    # ── Create the SageMaker Private Workforce ───────────────────────────────
    try:
        workforce_response = sagemaker_client.create_workforce(
            WorkforceName=f"my-review-workforce-{UNIQUE_SUFFIX}",
            CognitoConfig={
                "UserPool": COGNITO_POOL_ID,
                "ClientId": COGNITO_CLIENT_ID
            }
        )
        WORKFORCE_ARN = workforce_response["WorkforceArn"]
        print(f"Private workforce created: {WORKFORCE_ARN}")
    except sagemaker_client.exceptions.ResourceInUse:
        existing = sagemaker_client.list_workforces()["Workforces"]
        WORKFORCE_ARN = existing[0]["WorkforceArn"]
        print(f"Workforce already existed, reusing: {WORKFORCE_ARN}")

print(f"\nWORKFORCE_ARN     : {WORKFORCE_ARN}")
print(f"COGNITO_POOL_ID   : {COGNITO_POOL_ID}")
print(f"COGNITO_CLIENT_ID : {COGNITO_CLIENT_ID}")


Found existing private workforce — reusing: arn:aws:sagemaker:us-east-1:745404749079:workforce/default


  Cognito Pool ID : us-east-1_5ozXljgg7
  Cognito Client ID: 63qhkvk9bb91cufvhecufhlb86

WORKFORCE_ARN     : arn:aws:sagemaker:us-east-1:745404749079:workforce/default
COGNITO_POOL_ID   : us-east-1_5ozXljgg7
COGNITO_CLIENT_ID : 63qhkvk9bb91cufvhecufhlb86


> **What the cell above does automatically:** It provisions a Cognito User Pool, App Client, and User Group — the same resources the AWS Console creates "behind the scenes" the first time you set up a private workforce. If a workforce already exists in this region (only one is allowed per account/region), it is reused and its Cognito config is read back automatically. No manual copy-pasting of IDs is needed.

In [32]:
# ---------------------------------------------------------------
# Refresh workforce list (used by the work-team cell below)
# ---------------------------------------------------------------
existing_workforces = sagemaker_client.list_workforces()["Workforces"]
print(f"Workforces available: {len(existing_workforces)}")
for wf in existing_workforces:
    print(f"  - {wf['WorkforceName']}  ARN: {wf['WorkforceArn']}")


Workforces available: 1
  - default  ARN: arn:aws:sagemaker:us-east-1:745404749079:workforce/default


In [33]:
# ---------------------------------------------------------------
# Step 2: Create the Work Team using the Cognito IDs from Step 1
# ---------------------------------------------------------------
# This is equivalent to entering the reviewer email + org details
# under "Add workers" in the console.

# Determine the workforce name to attach the team to
workforce_name = (
    existing_workforces[0]["WorkforceName"]
    if existing_workforces
    else f"my-review-workforce-{UNIQUE_SUFFIX}"
)

# COGNITO_POOL_ID, COGNITO_CLIENT_ID, COGNITO_GROUP_NAME were set in Step 1
try:
    work_team_response = sagemaker_client.create_workteam(
        WorkteamName=f"my-review-team-{UNIQUE_SUFFIX}",
        WorkforceName=workforce_name,
        MemberDefinitions=[
            {
                "CognitoMemberDefinition": {
                    "UserPool":   COGNITO_POOL_ID,
                    "UserGroup":  COGNITO_GROUP_NAME,
                    "ClientId":   COGNITO_CLIENT_ID
                }
            }
        ],
        Description="Private review team for AI prediction approval/rejection",
        NotificationConfiguration={}
    )
    WORK_TEAM_ARN = work_team_response["WorkteamArn"]
    print(f"Work team created: {WORK_TEAM_ARN}")
except sagemaker_client.exceptions.ResourceInUse:
    # Team with this suffix already exists (e.g. kernel restart); fetch ARN
    teams = sagemaker_client.list_workteams()["Workteams"]
    WORK_TEAM_ARN = teams[0]["WorkteamArn"]
    print(f"Work team already exists, reusing: {WORK_TEAM_ARN}")

print(f"\nWORK_TEAM_ARN: {WORK_TEAM_ARN}")
print("""
NOTE: To add a real human reviewer, invite them to the Cognito User Pool
from the AWS Console > Cognito > User Pools > your pool > Users > Create user.
That one-time identity verification step cannot be scripted (by AWS design).
""")


Work team created: arn:aws:sagemaker:us-east-1:745404749079:workteam/private-crowd/my-review-team-49f4e565

WORK_TEAM_ARN: arn:aws:sagemaker:us-east-1:745404749079:workteam/private-crowd/my-review-team-49f4e565

NOTE: To add a real human reviewer, invite them to the Cognito User Pool
from the AWS Console > Cognito > User Pools > your pool > Users > Create user.
That one-time identity verification step cannot be scripted (by AWS design).



### Step 3: Create the Worker Task Template

This replaces pasting the `crowd-classifier` HTML into the console's **Template editor**. The HTML/Liquid template is identical to the one in the original guide.


In [34]:
# ---------------------------------------------------------------
# Create the custom worker task template (same HTML as the console lab)
# ---------------------------------------------------------------
worker_template_html = '''
<script src="https://assets.crowd.aws/crowd-html-elements.js"></script>
<crowd-form>
  <crowd-classifier
    name="category"
    categories="['Approve', 'Reject']"
    header="Review this AI prediction and approve or reject it">
    <classification-target>
      {{ task.input.taskObject }}
    </classification-target>
    <full-instructions header="Instructions">
      Review the AI prediction and select Approve or Reject.
    </full-instructions>
    <short-instructions>
      Approve or Reject the prediction.
    </short-instructions>
  </crowd-classifier>
</crowd-form>
'''

template_response = sagemaker_client.create_human_task_ui(
    HumanTaskUiName=f"ai-review-template-{UNIQUE_SUFFIX}",
    UiTemplate={"Content": worker_template_html}
)
HUMAN_TASK_UI_ARN = template_response["HumanTaskUiArn"]
print(f"Worker task template created: {HUMAN_TASK_UI_ARN}")


Worker task template created: arn:aws:sagemaker:us-east-1:745404749079:human-task-ui/ai-review-template-49f4e565


### Step 4: Create the Human Review Workflow (Flow Definition)

This replaces the final **"Create human review workflow"** screen in the console, tying together the IAM role, S3 output path, task template, and work team.


In [35]:
# ---------------------------------------------------------------
# Create the A2I human review workflow ("flow definition")
# ---------------------------------------------------------------
A2I_OUTPUT_S3_URI = f"s3://{BUCKET_NAME}/{A2I_OUTPUT_PREFIX}"

flow_definition_response = sagemaker_client.create_flow_definition(
    FlowDefinitionName=f"ai-review-workflow-{UNIQUE_SUFFIX}",
    HumanLoopConfig={
        "WorkteamArn": WORK_TEAM_ARN,
        "HumanTaskUiArn": HUMAN_TASK_UI_ARN,
        "TaskTitle": "Review AI Prediction",
        "TaskDescription": "Review and approve or reject the AI prediction result",
        "TaskCount": 1,
        "TaskAvailabilityLifetimeInSeconds": 864000,   # 10 days, console default
        "TaskTimeLimitInSeconds": 3600                  # 1 hour, console default
    },
    OutputConfig={"S3OutputPath": A2I_OUTPUT_S3_URI},
    RoleArn=a2i_role_arn
)
FLOW_DEFINITION_ARN = flow_definition_response["FlowDefinitionArn"]
print(f"A2I workflow (flow definition) created: {FLOW_DEFINITION_ARN}")


A2I workflow (flow definition) created: arn:aws:sagemaker:us-east-1:745404749079:flow-definition/ai-review-workflow-49f4e565


In [36]:
# ---------------------------------------------------------------
# Verify the workflow is Active (equivalent to checking workflow status)
# ---------------------------------------------------------------
def wait_for_flow_definition(arn, poll_seconds=15):
    while True:
        status = sagemaker_client.describe_flow_definition(
            FlowDefinitionName=arn.split("/")[-1]
        )["FlowDefinitionStatus"]
        print(f"  Flow definition status: {status}")
        if status == "Active":
            print("A2I human review workflow is Active.")
            break
        elif status == "Failed":
            raise Exception("Flow definition creation FAILED.")
        time.sleep(poll_seconds)

wait_for_flow_definition(FLOW_DEFINITION_ARN)
print(f"\nWorkflow ARN (save for use in applications): {FLOW_DEFINITION_ARN}")


  Flow definition status: Active
A2I human review workflow is Active.

Workflow ARN (save for use in applications): arn:aws:sagemaker:us-east-1:745404749079:flow-definition/ai-review-workflow-49f4e565


### (Optional) Trigger a Human Loop — Simulating a Low-Confidence AI Decision

In a full production pipeline, you'd start a **human loop** whenever Kendra/Personalize returns a low-confidence result. This is the equivalent of the original guide's note that *"A2I acts as a safety net... ensuring that uncertain AI predictions... are reviewed by a human worker."* The cell below shows how that integration works in code.


In [37]:
# ---------------------------------------------------------------
# (Optional) Trigger a Human Loop — simulate a low-confidence AI decision
# ---------------------------------------------------------------
a2i_runtime_client = session.client("sagemaker-a2i-runtime")

human_loop_name = f"review-loop-{UNIQUE_SUFFIX}"

# Safely extract top recommendation details
top_item  = rec_response["itemList"][0]
top_score = top_item.get("score", "N/A")

sample_prediction_input = {
    "taskObject": (
        f"User {TEST_USER_ID} was recommended Item {top_item['itemId']} "
        f"(confidence score: {top_score}). "
        f"Please approve or reject this recommendation."
    )
}

try:
    human_loop_response = a2i_runtime_client.start_human_loop(
        HumanLoopName=human_loop_name,
        FlowDefinitionArn=FLOW_DEFINITION_ARN,
        HumanLoopInput={"InputContent": json.dumps(sample_prediction_input)}
    )
    print(f"Human loop started: {human_loop_response['HumanLoopArn']}")
    print("The reviewer will see this task in their worker portal once they accept their invite.")
except Exception as e:
    print(f"Human loop note: {e}")
    print("This is expected if the work team has no confirmed members yet.")
    print("The workflow itself is Active and valid for lab validation purposes.")


Human loop started: arn:aws:sagemaker:us-east-1:745404749079:human-loop/review-loop-49f4e565
The reviewer will see this task in their worker portal once they accept their invite.


---
## Section 6: Verify End-to-End Output

This replaces the original guide's step: *"go to the configured S3 bucket and verify that the workflow output files are being generated successfully."*


In [38]:
# ---------------------------------------------------------------
# List objects in each S3 folder to confirm everything is in place
# ---------------------------------------------------------------
for prefix in [KENDRA_PREFIX, PERSONALIZE_PREFIX, A2I_OUTPUT_PREFIX]:
    print(f"\nContents of s3://{BUCKET_NAME}/{prefix}")
    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    for obj in response.get("Contents", []):
        print(f"  - {obj['Key']}  ({obj['Size']} bytes)")



Contents of s3://aws-ai-services-745404749079-us-east-1-49f4e565/kendra-docs/


  - kendra-docs/  (0 bytes)
  - kendra-docs/return-policy.txt  (656 bytes)

Contents of s3://aws-ai-services-745404749079-us-east-1-49f4e565/personalize-data/
  - personalize-data/  (0 bytes)
  - personalize-data/interactions.csv  (28146 bytes)

Contents of s3://aws-ai-services-745404749079-us-east-1-49f4e565/a2i-output/


  - a2i-output/  (0 bytes)
  - a2i-output/ai-review-workflow-49f4e565/temp/service-use.tmp  (2 bytes)


---
## Section 7: Delete and Clean Up

**This section mirrors the original guide's entire "Delete And Clean Up" chapter, end to end.** Run these cells in order (Personalize → A2I → Kendra → S3 → IAM) to avoid leaving any billable resources running.

> Just like the console version warns: **Amazon Kendra bills hourly** — don't skip the Kendra deletion step.


In [39]:
# =================================================================
# 7.1 Delete Amazon Personalize resources
# =================================================================
# Equivalent console steps: delete recommender -> delete dataset -> delete dataset group

# Delete the recommender first (must be deleted before the dataset group)
try:
    personalize_client.delete_recommender(recommenderArn=RECOMMENDER_ARN)
    print(f"Deletion requested for recommender: {RECOMMENDER_ARN}")
except Exception as e:
    print(f"Recommender deletion note: {e}")


Deletion requested for recommender: arn:aws:personalize:us-east-1:745404749079:recommender/product-recommender-49f4e565


In [40]:
# Poll until the recommender is fully deleted before continuing
def wait_for_deletion(describe_fn, **kwargs):
    while True:
        try:
            describe_fn(**kwargs)
            print("  Still deleting...")
            time.sleep(20)
        except Exception:
            print("  Resource deleted.")
            break

wait_for_deletion(personalize_client.describe_recommender, recommenderArn=RECOMMENDER_ARN)


  Still deleting...


  Still deleting...


  Still deleting...


  Still deleting...


  Still deleting...


...

  Still deleting...


  Still deleting...


  Still deleting...


  Still deleting...


  Resource deleted.


In [41]:
# Delete the Interactions dataset
try:
    personalize_client.delete_dataset(datasetArn=INTERACTIONS_DATASET_ARN)
    print(f"Deletion requested for dataset: {INTERACTIONS_DATASET_ARN}")
except Exception as e:
    print(f"Dataset deletion note: {e}")

wait_for_deletion(personalize_client.describe_dataset, datasetArn=INTERACTIONS_DATASET_ARN)


Deletion requested for dataset: arn:aws:personalize:us-east-1:745404749079:dataset/retail-recommendation-group-49f4e565/INTERACTIONS


  Still deleting...


  Resource deleted.


In [42]:
# Delete the schema
try:
    personalize_client.delete_schema(schemaArn=INTERACTIONS_SCHEMA_ARN)
    print(f"Schema deleted: {INTERACTIONS_SCHEMA_ARN}")
except Exception as e:
    print(f"Schema deletion note: {e}")


Schema deleted: arn:aws:personalize:us-east-1:745404749079:schema/retail-interactions-schema-49f4e565


In [43]:
# Delete the dataset group itself
try:
    personalize_client.delete_dataset_group(datasetGroupArn=DATASET_GROUP_ARN)
    print(f"Deletion requested for dataset group: {DATASET_GROUP_ARN}")
except Exception as e:
    print(f"Dataset group deletion note: {e}")

wait_for_deletion(personalize_client.describe_dataset_group, datasetGroupArn=DATASET_GROUP_ARN)
print("\nAll Amazon Personalize resources cleaned up.")


Deletion requested for dataset group: arn:aws:personalize:us-east-1:745404749079:dataset-group/retail-recommendation-group-49f4e565


  Still deleting...


  Still deleting...


  Resource deleted.

All Amazon Personalize resources cleaned up.


In [44]:
# =================================================================
# 7.2 Delete Amazon A2I Workflow & Worker Template
# =================================================================
# Equivalent console steps: delete ai-review-workflow -> delete ai-review-template

flow_def_name = FLOW_DEFINITION_ARN.split("/")[-1]
try:
    sagemaker_client.delete_flow_definition(FlowDefinitionName=flow_def_name)
    print(f"Deleted A2I flow definition: {flow_def_name}")
except Exception as e:
    print(f"Flow definition deletion note: {e}")

template_name = HUMAN_TASK_UI_ARN.split("/")[-1]
try:
    sagemaker_client.delete_human_task_ui(HumanTaskUiName=template_name)
    print(f"Deleted worker task template: {template_name}")
except Exception as e:
    print(f"Worker task template deletion note: {e}")

# Optionally also remove the work team (leave the shared private workforce
# itself intact, since it may be reused by other labs/workflows)
try:
    work_team_name = WORK_TEAM_ARN.split("/")[-1]
    sagemaker_client.delete_workteam(WorkteamName=work_team_name)
    print(f"Deleted work team: {work_team_name}")
except Exception as e:
    print(f"Work team deletion note: {e}")


Deleted A2I flow definition: ai-review-workflow-49f4e565


Deleted worker task template: ai-review-template-49f4e565


Deleted work team: my-review-team-49f4e565


In [45]:
# =================================================================
# 7.3 Delete Amazon Kendra resources
# =================================================================
# Equivalent console steps: delete data source -> delete index

try:
    kendra_client.delete_data_source(Id=KENDRA_DATASOURCE_ID, IndexId=KENDRA_INDEX_ID)
    print(f"Deletion requested for Kendra data source: {KENDRA_DATASOURCE_ID}")
except Exception as e:
    print(f"Data source deletion note: {e}")

time.sleep(15)  # allow the data source deletion to register before deleting the index

try:
    kendra_client.delete_index(Id=KENDRA_INDEX_ID)
    print(f"Deletion requested for Kendra index: {KENDRA_INDEX_ID}")
    print("NOTE: Kendra index deletion can take several minutes to complete in the background.")
except Exception as e:
    print(f"Kendra index deletion note: {e}")


Deletion requested for Kendra data source: 536b0593-c8b5-47e4-ae8f-63b184c5ba3b


Deletion requested for Kendra index: 6be3954d-1296-4e4b-86fe-7ae4799dd695
NOTE: Kendra index deletion can take several minutes to complete in the background.


In [46]:
# =================================================================
# 7.4 Delete the S3 bucket (empty it first, then delete)
# =================================================================
# Equivalent console steps: "Empty" bucket -> "Delete" bucket

# Step 1: Delete all objects in the bucket (S3 buckets must be empty to delete)
paginator = s3_client.get_paginator("list_objects_v2")
objects_to_delete = []
for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        objects_to_delete.append({"Key": obj["Key"]})

if objects_to_delete:
    # S3 delete_objects accepts up to 1000 keys per call
    for i in range(0, len(objects_to_delete), 1000):
        batch = objects_to_delete[i:i+1000]
        s3_client.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": batch})
    print(f"Deleted {len(objects_to_delete)} objects from bucket {BUCKET_NAME}")
else:
    print("Bucket is already empty.")

# Step 2: Delete the now-empty bucket
s3_client.delete_bucket(Bucket=BUCKET_NAME)
print(f"Bucket deleted: {BUCKET_NAME}")


Deleted 7 objects from bucket aws-ai-services-745404749079-us-east-1-49f4e565


Bucket deleted: aws-ai-services-745404749079-us-east-1-49f4e565


In [47]:
# =================================================================
# 7.5 (Optional) Delete IAM roles created for this lab
# =================================================================
# Equivalent to manually removing the temporary IAM roles, for full cleanliness.
# Each role must have its attached/inline policies removed before deletion.

def delete_role_completely(role_name):
    try:
        # Detach managed policies
        attached = iam_client.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]
        for policy in attached:
            iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy["PolicyArn"])

        # Delete inline policies
        inline = iam_client.list_role_policies(RoleName=role_name)["PolicyNames"]
        for policy_name in inline:
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)

        # Delete the role itself
        iam_client.delete_role(RoleName=role_name)
        print(f"Deleted IAM role: {role_name}")
    except Exception as e:
        print(f"Role deletion note for {role_name}: {e}")

delete_role_completely(f"AmazonKendra-{REGION}-role-{UNIQUE_SUFFIX}")
delete_role_completely(f"AmazonPersonalize-role-{UNIQUE_SUFFIX}")
delete_role_completely(f"AmazonA2I-role-{UNIQUE_SUFFIX}")

print("\nAll lab resources have been cleaned up.")


Deleted IAM role: AmazonKendra-us-east-1-role-49f4e565


Deleted IAM role: AmazonPersonalize-role-49f4e565


Deleted IAM role: AmazonA2I-role-49f4e565

All lab resources have been cleaned up.


---
## Section 8: Bonus — Verify Resource Status via AWS CLI / boto3

The original guide includes a "Bonus: Verify Service Status" section using AWS CLI commands. Here are their direct `boto3` equivalents, useful for confirming cleanup succeeded.


In [48]:
# ---------------------------------------------------------------
# Verify no Kendra indices remain
# ---------------------------------------------------------------
print("Kendra indices remaining:")
print(kendra_client.list_indices())


Kendra indices remaining:
{'IndexConfigurationSummaryItems': [{'Name': 'business-search-index-0955156e', 'Id': '392e4de6-fff9-4cec-a7b3-5cfaf9d8cb09', 'Edition': 'DEVELOPER_EDITION', 'CreatedAt': datetime.datetime(2026, 6, 21, 10, 22, 8, 105000, tzinfo=tzlocal()), 'UpdatedAt': datetime.datetime(2026, 6, 21, 10, 22, 8, 105000, tzinfo=tzlocal()), 'Status': 'ACTIVE'}, {'Name': 'business-search-index-338b1835', 'Id': 'd9362a4f-19cd-4ea3-8637-391b1f90671a', 'Edition': 'DEVELOPER_EDITION', 'CreatedAt': datetime.datetime(2026, 6, 21, 7, 27, 57, 609000, tzinfo=tzlocal()), 'UpdatedAt': datetime.datetime(2026, 6, 21, 7, 27, 57, 609000, tzinfo=tzlocal()), 'Status': 'ACTIVE'}, {'Name': 'business-search-index-ed35dc91', 'Id': 'e4cf667d-d6b7-4dd4-9726-8aeee9fc064b', 'Edition': 'DEVELOPER_EDITION', 'CreatedAt': datetime.datetime(2026, 6, 21, 10, 32, 30, 994000, tzinfo=tzlocal()), 'UpdatedAt': datetime.datetime(2026, 6, 21, 10, 32, 30, 994000, tzinfo=tzlocal()), 'Status': 'ACTIVE'}], 'ResponseMetadata

In [49]:
# ---------------------------------------------------------------
# Verify no Personalize dataset groups remain
# ---------------------------------------------------------------
print("Personalize dataset groups remaining:")
print(personalize_client.list_dataset_groups())


Personalize dataset groups remaining:


{'datasetGroups': [], 'ResponseMetadata': {'RequestId': '3c0edf8f-81f0-4fe0-9d35-180b84672eba', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 21 Jun 2026 12:28:37 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '20', 'connection': 'keep-alive', 'x-amzn-requestid': '3c0edf8f-81f0-4fe0-9d35-180b84672eba', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'cache-control': 'no-cache', 'x-content-type-options': 'nosniff'}, 'RetryAttempts': 0}}


In [50]:
# ---------------------------------------------------------------
# Verify no Personalize recommenders remain
# ---------------------------------------------------------------
print("Personalize recommenders remaining:")
print(personalize_client.list_recommenders())


Personalize recommenders remaining:


{'recommenders': [], 'ResponseMetadata': {'RequestId': 'ef9658dd-882f-4189-8869-2f2d3faf344a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 21 Jun 2026 12:28:42 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '19', 'connection': 'keep-alive', 'x-amzn-requestid': 'ef9658dd-882f-4189-8869-2f2d3faf344a', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'cache-control': 'no-cache', 'x-content-type-options': 'nosniff'}, 'RetryAttempts': 0}}


In [51]:
# ---------------------------------------------------------------
# Verify no A2I flow definitions remain
# ---------------------------------------------------------------
print("A2I flow definitions remaining:")
print(sagemaker_client.list_flow_definitions())


A2I flow definitions remaining:


{'FlowDefinitionSummaries': [], 'ResponseMetadata': {'RequestId': '957777c6-2193-4407-90a9-2d2671df4358', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '957777c6-2193-4407-90a9-2d2671df4358', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '30', 'date': 'Sun, 21 Jun 2026 12:28:47 GMT'}, 'RetryAttempts': 0}}


In [52]:
# ---------------------------------------------------------------
# Verify no A2I worker task templates remain
# ---------------------------------------------------------------
print("A2I worker task templates remaining:")
print(sagemaker_client.list_human_task_uis())


A2I worker task templates remaining:


{'HumanTaskUiSummaries': [], 'ResponseMetadata': {'RequestId': 'e2025eb3-87c0-45f5-8aad-9f64c5ce5cfc', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'e2025eb3-87c0-45f5-8aad-9f64c5ce5cfc', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '27', 'date': 'Sun, 21 Jun 2026 12:28:50 GMT'}, 'RetryAttempts': 0}}


---
## Troubleshooting

### `InvalidS3ObjectException` / Access Denied errors

**Cause:** The IAM role used by a given service does not have permission to read the S3 object, or the bucket policy does not explicitly allow that service's principal (e.g. `personalize.amazonaws.com`, `kendra.amazonaws.com`) to call `s3:GetObject`.

**Fix checklist:**
- Confirm the bucket policy in Section 1 was applied successfully (re-run `s3_client.get_bucket_policy(Bucket=BUCKET_NAME)` to inspect it).
- Confirm the relevant IAM role has `AmazonS3FullAccess` attached (check with `iam_client.list_attached_role_policies(RoleName=...)`).
- Confirm the S3 key path matches exactly — S3 keys are case-sensitive.
- Confirm all clients are using the same `REGION` variable throughout the notebook.

### Kendra sync shows one "failed" file

This is expected and matches the original guide's note: Kendra's S3 connector sometimes also attempts to process an internal metadata object it creates automatically. As long as your actual document (`return-policy.txt`) shows as successfully indexed in the sync history, the lab is working correctly.

### A2I workforce creation fails

This usually means a private workforce already exists in this region (only one is allowed per account/region) — use the "reuse existing workforce" cell in Section 5 instead of creating a new one.

---

## Summary

In this notebook, you built a complete, code-based **Intelligent Business Assistant** pipeline on AWS:

1. Provisioned an **S3 bucket** with organized prefixes and a least-privilege bucket policy
2. Created **IAM roles** scoped to Kendra, Personalize, and A2I (SageMaker)
3. Built an **Amazon Kendra** index, connected it to S3, synced documents, and tested natural-language search
4. Built an **Amazon Personalize** dataset group, imported interaction data, trained a "Recommended for you" recommender, and tested live recommendations
5. Built an **Amazon Augmented AI (A2I)** human review workflow with a custom approve/reject template and private workforce, and simulated routing a low-confidence prediction to a human reviewer
6. **Cleaned up every single resource** created, mirroring the original guide's emphasis on avoiding ongoing charges

This mirrors the same enterprise AI architecture used by large retailers — search, personalization, and human-in-the-loop review — but expressed entirely as **reusable, version-controllable Python code** instead of manual console clicks, which is how these systems are actually deployed and maintained in production via Infrastructure-as-Code.
